In [ ]:
import glob
import time
from sokoban import Warehouse

import multiprocessing as mp
import multiprocessing.queues as mpq

from typing import Tuple, Callable, Dict

## This Testing File

This file shows how to automate testing across all warehouses **with a timeout option** for warehouses that take longer than 3 minutes. It may assist you in performing a more widescale evaluation.

In [ ]:
#Import the mySokobanSolver.py file to import all functions
from mySokobanSolver import *

## Visualise results

In [ ]:
def test_warehouse(problem_file, macro = False):
    '''
    This function will test the performance of your warehouse for either macro or elem solutions and return the result.
    You can check if this solution works with your gui, or by cleverly using the check_action_seq function.
    '''

    wh = Warehouse()
    wh.load_warehouse(problem_file)

    if macro:
        student_answer =  solve_sokoban_macro(wh)
    else:
        student_answer = solve_sokoban_elem(wh)
        
    return student_answer
#.............................................................


In [ ]:
all_warehouses = sorted(glob.glob('warehouses/*.txt'))

def warehouse_timeout(args: Tuple[object], q: mp.Queue):
    #Do not alter this code.
    q.put(test_warehouse(*args))
    
def test_with_timeout(problem_file, macro = False, timeout = 180):
    """
    This function tests on a warehouse with the ability to timeout after a specified number of seconds. 

    Parameters:
    problem_file (str): directory of a warehouse
    macro (bool): indicates whether to use the macro solver. If false, will use the elem solver
    timeout (int): The number of seconds the solver can run without timing out.

    Returns:
    The solver solution or the string "Timed out"
    """
    
    q_worker = mp.Queue()
    proc = mp.Process(target=warehouse_timeout, args=((problem_file,macro), q_worker))
    proc.start()
    try:
        res = q_worker.get(timeout=timeout)
    except mpq.Empty:
        proc.terminate()
        res = "Timed out"
    return res

for problem_file in all_warehouses:
    print(f'Testing {problem_file}')
    
    print("Testing Elem Solver:")
    s = time.time()
    a = test_with_timeout(problem_file)
    print(f'Answer: {a}')
    print(f'Time taken: {time.time()-s :.3f} seconds')          
    
    print("Testing Macro Solver:")
    s = time.time()
    a = test_with_timeout(problem_file, True)
    print(f'Answer for Macro Solver: {a}')
    print(f'Time taken: {time.time()-s :.3f} seconds')          